# Complete CamemBERT Workflow Demonstration

This notebook is a single, runnable guide for the `BERT/` pipeline. It uses the existing project modules instead of reimplementing the pipeline.

The task is binary text classification:

- `femme -> 0`
- `homme -> 1`

The model fine-tunes HuggingFace `camembert-base` with `CamembertForSequenceClassification`. Long documents are split into 512-token chunks with the CamemBERT tokenizer.

In [2]:
#test

## 1. Repository Setup

This cell makes the notebook work when launched either from the repository root or from inside `BERT/`. It also creates the expected output folders.

In [3]:
from __future__ import annotations

from argparse import Namespace
from contextlib import contextmanager
from pathlib import Path
import importlib
import importlib.util
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import HTML, Image, display

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "BERT").exists():
    PROJECT_ROOT = CURRENT_DIR
    BERT_DIR = PROJECT_ROOT / "BERT"
else:
    BERT_DIR = CURRENT_DIR
    PROJECT_ROOT = BERT_DIR.parent

if str(BERT_DIR) not in sys.path:
    sys.path.insert(0, str(BERT_DIR))

for folder in [
    BERT_DIR / "outputs" / "checkpoints",
    BERT_DIR / "outputs" / "models",
    BERT_DIR / "outputs" / "logs",
    BERT_DIR / "artifacts",
    BERT_DIR / "vis",
]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"BERT directory: {BERT_DIR}")
print(f"Python executable: {sys.executable}")

Project root: /
BERT directory: /content
Python executable: /usr/bin/python3


## 2. Imports and Main Configuration

The notebook imports the existing modules from `BERT/`. These modules contain the real implementation used by the command-line scripts.

In [ ]:
import config as config_module
import data_loader as data_loader_module
import model as model_module
import utils as utils_module
import train as train_pipeline
import evaluate as evaluate_pipeline
import experiment as experiment_pipeline

for module in [
    config_module,
    data_loader_module,
    model_module,
    utils_module,
    train_pipeline,
    evaluate_pipeline,
    experiment_pipeline,
]:
    importlib.reload(module)

from config import ID2LABEL, LABEL2ID, ModelConfig, PathConfig, TrainingConfig
from data_loader import chunk_text, create_dataloader, describe_chunk_selection, documents_to_frame, load_splits
from model import create_tokenizer
from utils import load_json

paths = PathConfig()
model_defaults = ModelConfig()
train_defaults = TrainingConfig()

DATA_DIR = paths.data_dir
MODEL_NAME = model_defaults.model_name
MAX_LENGTH = model_defaults.max_length
NUM_CHUNKS_HOMME = model_defaults.num_chunks_homme
NUM_CHUNKS_FEMME = model_defaults.num_chunks_femme
BATCH_SIZE = train_defaults.batch_size
EVAL_BATCH_SIZE = train_defaults.eval_batch_size
EPOCHS = train_defaults.epochs
LEARNING_RATE = train_defaults.learning_rate
PATIENCE = train_defaults.patience

CHECKPOINT_DIR = paths.checkpoint_dir
BEST_MODEL_DIR = paths.best_model_dir
LOG_DIR = paths.log_dir
ARTIFACT_DIR = paths.artifact_dir
VIS_DIR = paths.vis_dir

config_table = pd.DataFrame(
    [
        ("model_name", MODEL_NAME),
        ("max_length", MAX_LENGTH),
        ("num_chunks_homme", NUM_CHUNKS_HOMME),
        ("num_chunks_femme", NUM_CHUNKS_FEMME),
        ("batch_size", BATCH_SIZE),
        ("eval_batch_size", EVAL_BATCH_SIZE),
        ("epochs", EPOCHS),
        ("learning_rate", LEARNING_RATE),
        ("patience", PATIENCE),
        ("data_dir", DATA_DIR),
        ("best_model_dir", BEST_MODEL_DIR),
        ("checkpoint_dir", CHECKPOINT_DIR),
        ("log_dir", LOG_DIR),
        ("artifact_dir", ARTIFACT_DIR),
        ("vis_dir", VIS_DIR),
    ],
    columns=["parameter", "value"],
)
display(config_table)

The helpers below let this notebook call existing script entry points, such as `evaluate.main()` or `experiment.main()`, with notebook-defined arguments.

In [ ]:
@contextmanager
def patched_argv(argv: list[str]):
    old_argv = sys.argv[:]
    sys.argv = argv
    try:
        yield
    finally:
        sys.argv = old_argv


def clean_cli_args(args: list[object]) -> list[str]:
    cleaned = []
    index = 0
    while index < len(args):
        item = args[index]
        next_item = args[index + 1] if index + 1 < len(args) else None
        if isinstance(item, str) and item.startswith("--") and next_item is None:
            index += 2
            continue
        if item is not None:
            cleaned.append(str(item))
        index += 1
    return cleaned


def run_module_main(module, args: list[object]) -> None:
    argv = [str(Path(module.__file__).name), *clean_cli_args(args)]
    with patched_argv(argv):
        module.main()


def dependency_available(package_name: str) -> bool:
    return importlib.util.find_spec(package_name) is not None


print("Notebook helper functions ready.")

## 3. Dataset Exploration

The dataset loader reads the existing `train`, `val`, and `test` folders. Labels are inferred with the same convention used elsewhere in the project.

In [ ]:
train_docs, val_docs, test_docs = load_splits(DATA_DIR)
splits = {"train": train_docs, "val": val_docs, "test": test_docs}

split_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "n_documents": len(documents),
            "femme": sum(doc.label == "femme" for doc in documents),
            "homme": sum(doc.label == "homme" for doc in documents),
        }
        for split_name, documents in splits.items()
    ]
)
display(split_summary)

sample_rows = []
for split_name, documents in splits.items():
    for doc in documents[:2]:
        sample_rows.append(
            {
                "split": split_name,
                "path": doc.path,
                "label": doc.label,
                "label_id": doc.label_id,
                "text_preview": doc.text.replace("\n", " ")[:300],
            }
        )

display(pd.DataFrame(sample_rows))

In [ ]:
class_distribution_path = VIS_DIR / "dataset_class_distribution.png"

plot_df = split_summary.set_index("split")[["femme", "homme"]]
ax = plot_df.plot(kind="bar", figsize=(8, 5))
ax.set_title("Class distribution by split")
ax.set_xlabel("Split")
ax.set_ylabel("Number of documents")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(class_distribution_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved class distribution plot: {class_distribution_path}")

## 4. Tokenization and Chunking Demonstration

CamemBERT has a maximum sequence length of 512 tokens. Long documents are split into multiple chunks; every chunk inherits the document label.

In [ ]:
tokenizer = create_tokenizer(MODEL_NAME)
all_documents = train_docs + val_docs + test_docs
demo_doc = max(all_documents, key=lambda doc: len(doc.text))
demo_chunks = chunk_text(
    demo_doc.text,
    tokenizer,
    max_length=MAX_LENGTH,
    label=demo_doc.label,
    num_chunks_homme=NUM_CHUNKS_HOMME,
    num_chunks_femme=NUM_CHUNKS_FEMME,
    seed=train_defaults.seed,
    sample_key=demo_doc.path,
)

input_ids_tensor = torch.tensor([chunk["input_ids"] for chunk in demo_chunks], dtype=torch.long)
attention_mask_tensor = torch.tensor([chunk["attention_mask"] for chunk in demo_chunks], dtype=torch.long)

print(f"Document path: {demo_doc.path}")
print(f"Label: {demo_doc.label} -> {demo_doc.label_id}")
print(f"Original character length: {len(demo_doc.text)}")
print(describe_chunk_selection(NUM_CHUNKS_HOMME, NUM_CHUNKS_FEMME))
print(f"Number of selected chunks: {len(demo_chunks)}")
print(f"input_ids shape: {tuple(input_ids_tensor.shape)}")
print(f"attention_mask shape: {tuple(attention_mask_tensor.shape)}")
print("Decoded preview of first chunk:")
print(tokenizer.decode(demo_chunks[0]["input_ids"][:100]))

## 5. Training

Edit the parameters below before launching training. The next cell calls `train_pipeline.run_training(args)` from `BERT/train.py`; it does not reimplement the training loop.

In [ ]:
# Editable training parameters
DATA_DIR = paths.data_dir
MODEL_NAME = model_defaults.model_name
MAX_LENGTH = model_defaults.max_length
NUM_CHUNKS_HOMME = model_defaults.num_chunks_homme
NUM_CHUNKS_FEMME = model_defaults.num_chunks_femme
BATCH_SIZE = train_defaults.batch_size
EVAL_BATCH_SIZE = train_defaults.eval_batch_size
EPOCHS = train_defaults.epochs
LEARNING_RATE = train_defaults.learning_rate
WEIGHT_DECAY = train_defaults.weight_decay
WARMUP_RATIO = train_defaults.warmup_ratio
MAX_GRAD_NORM = train_defaults.max_grad_norm
PATIENCE = train_defaults.patience
NUM_WORKERS = train_defaults.num_workers
SEED = train_defaults.seed
NO_AMP = False

CHECKPOINT_DIR = paths.checkpoint_dir
BEST_MODEL_DIR = paths.best_model_dir
LOG_DIR = paths.log_dir
ARTIFACT_DIR = paths.artifact_dir

training_config = pd.DataFrame(
    [
        ("DATA_DIR", DATA_DIR),
        ("MODEL_NAME", MODEL_NAME),
        ("MAX_LENGTH", MAX_LENGTH),
        ("NUM_CHUNKS_HOMME", NUM_CHUNKS_HOMME),
        ("NUM_CHUNKS_FEMME", NUM_CHUNKS_FEMME),
        ("BATCH_SIZE", BATCH_SIZE),
        ("EVAL_BATCH_SIZE", EVAL_BATCH_SIZE),
        ("EPOCHS", EPOCHS),
        ("LEARNING_RATE", LEARNING_RATE),
        ("PATIENCE", PATIENCE),
        ("CHECKPOINT_DIR", CHECKPOINT_DIR),
        ("BEST_MODEL_DIR", BEST_MODEL_DIR),
    ],
    columns=["parameter", "value"],
)
display(training_config)

In [ ]:
training_args = Namespace(
    data_dir=Path(DATA_DIR),
    model_name=str(MODEL_NAME),
    max_length=MAX_LENGTH,
    num_chunks_homme=NUM_CHUNKS_HOMME,
    num_chunks_femme=NUM_CHUNKS_FEMME,
    batch_size=BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=MAX_GRAD_NORM,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    no_amp=NO_AMP,
    checkpoint_dir=Path(CHECKPOINT_DIR),
    best_model_dir=Path(BEST_MODEL_DIR),
    log_dir=Path(LOG_DIR),
    artifact_dir=Path(ARTIFACT_DIR),
)

training_metrics = train_pipeline.run_training(training_args)
display(pd.DataFrame(training_metrics["history"]))

## 6. Training History Visualization

The training script writes `BERT/outputs/logs/history.csv`. This cell plots loss and validation metrics from that file.

In [ ]:
history_path = LOG_DIR / "history.csv"
history_df = pd.read_csv(history_path)
display(history_df)

loss_plot_path = VIS_DIR / "training_loss.png"
metrics_plot_path = VIS_DIR / "validation_metrics.png"

ax = history_df.plot(x="epoch", y=["train_loss", "val_loss"], marker="o", figsize=(8, 5))
ax.set_title("Training and validation loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
plt.tight_layout()
plt.savefig(loss_plot_path, dpi=150, bbox_inches="tight")
plt.show()

metric_columns = [column for column in ["val_accuracy", "val_f1"] if column in history_df.columns]
ax = history_df.plot(x="epoch", y=metric_columns, marker="o", figsize=(8, 5))
ax.set_title("Validation metrics")
ax.set_xlabel("Epoch")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(metrics_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved loss plot: {loss_plot_path}")
print(f"Saved validation metrics plot: {metrics_plot_path}")

## 7. Evaluation

Evaluation loads the best saved model, predicts the test split at document level, saves predictions, and updates `BERT/artifacts/metrics.json`.

In [ ]:
run_module_main(
    evaluate_pipeline,
    [
        "--data_dir", DATA_DIR,
        "--model_dir", BEST_MODEL_DIR,
        "--artifact_dir", ARTIFACT_DIR,
        "--vis_dir", VIS_DIR,
        "--max_length", MAX_LENGTH,
        "--num_chunks_homme", NUM_CHUNKS_HOMME,
        "--num_chunks_femme", NUM_CHUNKS_FEMME,
        "--seed", SEED,
        "--batch_size", EVAL_BATCH_SIZE,
    ],
)

metrics = load_json(ARTIFACT_DIR / "metrics.json")
test_metrics = metrics.get("test", {})
metric_rows = [
    ("accuracy", test_metrics.get("accuracy")),
    ("precision", test_metrics.get("precision")),
    ("recall", test_metrics.get("recall")),
    ("f1", test_metrics.get("f1")),
    ("roc_auc", test_metrics.get("roc_auc")),
]
display(pd.DataFrame(metric_rows, columns=["metric", "value"]))

if "classification_report" in test_metrics:
    display(pd.DataFrame(test_metrics["classification_report"]).T)

confusion_matrix_path = VIS_DIR / "confusion_matrix.png"
roc_curve_path = VIS_DIR / "roc_curve.png"
if confusion_matrix_path.exists():
    display(Image(filename=str(confusion_matrix_path)))
if roc_curve_path.exists():
    display(Image(filename=str(roc_curve_path)))

In [ ]:
predictions_path = ARTIFACT_DIR / "test_predictions.csv"
predictions_df = pd.read_csv(predictions_path)
display(predictions_df.head(20))
print(f"Saved predictions: {predictions_path}")

## 8. LIME Explainability

LIME explains individual predictions by perturbing text and observing how the model probability changes. This uses `BERT/explain_lime.py`.

In [ ]:
LIME_N_EXAMPLES = 10
LIME_N_TERMS = 20

if not dependency_available("lime"):
    print("LIME is not installed. Install it with:")
    print("pip install lime")
else:
    explain_lime_pipeline = importlib.import_module("explain_lime")
    run_module_main(
        explain_lime_pipeline,
        [
            "--data_dir", DATA_DIR,
            "--model_dir", BEST_MODEL_DIR,
            "--n_examples", LIME_N_EXAMPLES,
            "--n_terms", LIME_N_TERMS,
            "--artifact_dir", ARTIFACT_DIR,
            "--vis_dir", VIS_DIR,
            "--max_length", MAX_LENGTH,
            "--num_chunks_homme", NUM_CHUNKS_HOMME,
            "--num_chunks_femme", NUM_CHUNKS_FEMME,
            "--seed", SEED,
            "--batch_size", EVAL_BATCH_SIZE,
        ],
    )

    lime_results_path = ARTIFACT_DIR / "lime_results.csv"
    lime_html_path = VIS_DIR / "lime_local_explanation.html"
    lime_femme_path = VIS_DIR / "lime_top_femme_terms.png"
    lime_homme_path = VIS_DIR / "lime_top_homme_terms.png"

    if lime_results_path.exists():
        display(pd.read_csv(lime_results_path).head(30))
    if lime_html_path.exists():
        display(HTML(lime_html_path.read_text(encoding="utf-8")))
    if lime_femme_path.exists():
        display(Image(filename=str(lime_femme_path)))
    if lime_homme_path.exists():
        display(Image(filename=str(lime_homme_path)))

## 9. SHAP Explainability

SHAP can be slower than LIME. This demonstration uses a small default number of examples and handles missing dependencies gracefully.

In [ ]:
SHAP_N_EXAMPLES = 5
SHAP_N_TERMS = 20

if not dependency_available("shap"):
    print("SHAP is not installed. Install it with:")
    print("pip install shap")
else:
    explain_shap_pipeline = importlib.import_module("explain_shap")
    try:
        run_module_main(
            explain_shap_pipeline,
            [
                "--data_dir", DATA_DIR,
                "--model_dir", BEST_MODEL_DIR,
                "--n_examples", SHAP_N_EXAMPLES,
                "--n_terms", SHAP_N_TERMS,
                "--artifact_dir", ARTIFACT_DIR,
                "--vis_dir", VIS_DIR,
                "--max_length", MAX_LENGTH,
                "--num_chunks_homme", NUM_CHUNKS_HOMME,
                "--num_chunks_femme", NUM_CHUNKS_FEMME,
                "--seed", SEED,
                "--batch_size", EVAL_BATCH_SIZE,
            ],
        )
    except Exception as exc:
        print("SHAP execution failed or was interrupted. The rest of the notebook can continue.")
        print(f"Error: {exc}")

    shap_local_path = ARTIFACT_DIR / "shap_local.csv"
    shap_global_path = ARTIFACT_DIR / "shap_global.csv"
    shap_local_plot = VIS_DIR / "shap_local_explanation.png"
    shap_summary_plot = VIS_DIR / "shap_summary.png"

    if shap_local_path.exists():
        display(pd.read_csv(shap_local_path).head(30))
    if shap_global_path.exists():
        display(pd.read_csv(shap_global_path).head(30))
    if shap_local_plot.exists():
        display(Image(filename=str(shap_local_plot)))
    if shap_summary_plot.exists():
        display(Image(filename=str(shap_summary_plot)))

## 10. Experiment Comparison

The experiment framework compares model variants and explainability methods. It appends results to `BERT/artifacts/experiment_results.csv`.

In [ ]:
EXPERIMENT_N_EXAMPLES = 5
EXPERIMENT_N_TERMS = 20
experiment_results_path = ARTIFACT_DIR / "experiment_results.csv"

experiment_jobs = [
    ("pretrained", "lime"),
    ("finetuned", "lime"),
    ("finetuned", "shap"),
]

for model_type, method in experiment_jobs:
    if method == "lime" and not dependency_available("lime"):
        print("Skipping LIME experiment because lime is missing. Install it with: pip install lime")
        continue
    if method == "shap" and not dependency_available("shap"):
        print("Skipping SHAP experiment because shap is missing. Install it with: pip install shap")
        continue

    try:
        run_module_main(
            experiment_pipeline,
            [
                "--model", model_type,
                "--method", method,
                "--data_dir", DATA_DIR,
                "--model_name", MODEL_NAME,
                "--finetuned_model_dir", BEST_MODEL_DIR,
                "--n_examples", EXPERIMENT_N_EXAMPLES,
                "--n_terms", EXPERIMENT_N_TERMS,
                "--max_length", MAX_LENGTH,
                "--num_chunks_homme", NUM_CHUNKS_HOMME,
                "--num_chunks_femme", NUM_CHUNKS_FEMME,
                "--seed", SEED,
                "--batch_size", EVAL_BATCH_SIZE,
                "--output_path", experiment_results_path,
            ],
        )
    except Exception as exc:
        print(f"Experiment {model_type} + {method} failed. The notebook will continue.")
        print(f"Error: {exc}")

if experiment_results_path.exists():
    experiment_df = pd.read_csv(experiment_results_path)
    display(experiment_df.tail(30))

    comparison_plot_path = VIS_DIR / "experiment_confidence_comparison.png"
    comparison = (
        experiment_df.groupby(["model_type", "explainability_method"], as_index=False)["confidence"]
        .mean()
        .sort_values(["model_type", "explainability_method"])
    )
    if not comparison.empty:
        labels = comparison["model_type"] + " + " + comparison["explainability_method"]
        ax = comparison.assign(label=labels).plot(kind="bar", x="label", y="confidence", figsize=(9, 5), legend=False)
        ax.set_title("Mean confidence by experiment")
        ax.set_xlabel("Experiment")
        ax.set_ylabel("Mean confidence")
        ax.set_ylim(0, 1)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(comparison_plot_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved experiment comparison plot: {comparison_plot_path}")

## 11. Final Summary

This section lists the main generated outputs. Paths marked `False` have not been generated yet, usually because the related section has not been executed or an optional dependency is missing.

In [ ]:
output_paths = {
    "best_model": BEST_MODEL_DIR,
    "checkpoints": CHECKPOINT_DIR,
    "training_history": LOG_DIR / "history.csv",
    "metrics": ARTIFACT_DIR / "metrics.json",
    "test_predictions": ARTIFACT_DIR / "test_predictions.csv",
    "lime_results": ARTIFACT_DIR / "lime_results.csv",
    "lime_html": VIS_DIR / "lime_local_explanation.html",
    "lime_femme_terms": VIS_DIR / "lime_top_femme_terms.png",
    "lime_homme_terms": VIS_DIR / "lime_top_homme_terms.png",
    "shap_local": ARTIFACT_DIR / "shap_local.csv",
    "shap_global": ARTIFACT_DIR / "shap_global.csv",
    "shap_local_plot": VIS_DIR / "shap_local_explanation.png",
    "shap_summary_plot": VIS_DIR / "shap_summary.png",
    "experiment_results": ARTIFACT_DIR / "experiment_results.csv",
    "confusion_matrix": VIS_DIR / "confusion_matrix.png",
    "roc_curve": VIS_DIR / "roc_curve.png",
    "visualizations_dir": VIS_DIR,
}

summary_df = pd.DataFrame(
    [
        {"artifact": name, "path": path, "exists": Path(path).exists()}
        for name, path in output_paths.items()
    ]
)
display(summary_df)

print("Reminder: explanation outputs describe what this classifier learned from this dataset, not universal writing rules.")